# Fusion photo-z — train on Colab (logs to MLflow)

MLP + MDN(K=5) over the 83-d fusion input `[3 tabular base preds | 16 absence mask | 64 CNN embedding]`,
precomputed per (objid, missing-pattern) in `fusion_input_v4.parquet`. Early-stops on the **full-feature**
val subset (comparable to the tabular 0.0127 / CNN 0.0122 baselines); reports val σ_MAD by missing bucket.


In [ ]:
!pip -q install mlflow

In [ ]:
# --- MLflow connect (paste token at the prompt; it is NOT stored in the notebook) ---
import os, getpass, mlflow, mlflow.tensorflow
os.environ["MLFLOW_TRACKING_URI"] = "https://146-148-10-86.sslip.io"
os.environ["MLFLOW_TRACKING_TOKEN"] = getpass.getpass("MLflow token: ")
mlflow.set_experiment("fusion")
mlflow.tensorflow.autolog(log_models=False, log_datasets=False, checkpoint=False)
print("MLflow ->", os.environ["MLFLOW_TRACKING_URI"], "| experiment: fusion")

In [ ]:
# --- get the precomputed fusion inputs from GCS ---
from google.colab import auth; auth.authenticate_user()          # bucket access via your Google account
!gsutil -q cp gs://macrocosm-lewagon/data/fusion_input_v4.parquet /content/fusion_input_v4.parquet
import pandas as pd
df = pd.read_parquet("/content/fusion_input_v4.parquet")
print(df.shape); print(df["split"].value_counts().to_dict())

In [ ]:
# --- fusion model + MDN helpers (self-contained) ---
import numpy as np, tensorflow as tf
from tensorflow.keras import layers as L, Model, Input, regularizers

def mdn_nll(K):
    def loss(y_true, y_pred):
        pi, mu, sig = y_pred[:, :K], y_pred[:, K:2*K], y_pred[:, 2*K:]
        y = tf.expand_dims(y_true, 1)
        lc = (tf.math.log(pi+1e-8) - 0.5*tf.math.log(2*np.pi*sig**2+1e-8) - 0.5*((y-mu)/(sig+1e-8))**2)
        return tf.reduce_mean(-tf.reduce_logsumexp(lc, axis=1))
    return loss

def mdn_point(raw):                          # log1p(z)-space point estimate (mu of max-pi component)
    raw = np.asarray(raw)
    if raw.ndim == 2 and raw.shape[1] > 1 and raw.shape[1] % 3 == 0:
        K = raw.shape[1]//3; pi, mu = raw[:, :K], raw[:, K:2*K]
        return mu[np.arange(len(mu)), pi.argmax(1)]
    return raw.ravel()

def build_fusion(n_in, hidden=(128,128,64), mdn=5, l2=1e-4, drop=0.3):
    reg = regularizers.l2(l2)
    inp = Input((n_in,), name="fusion_in"); x = L.BatchNormalization()(inp)
    for i, h in enumerate(hidden):
        x = L.Dense(h, activation="relu", kernel_regularizer=reg, name=f"fc{i+1}")(x)
        x = L.BatchNormalization()(x); x = L.Dropout(drop)(x)
    pi  = L.Dense(mdn, activation="softmax", name="mdn_pi")(x)
    mu  = L.Dense(mdn, name="mdn_mu")(x)
    sig = L.Dense(mdn, activation="exponential", name="mdn_sigma")(x)
    return Model(inp, L.Concatenate(name="z")([pi, mu, sig]), name=f"fusion-mdn{mdn}")

def delta_z(zt, zp): zt, zp = np.asarray(zt,float), np.asarray(zp,float); return (zp-zt)/(1+zt)
def sigma_mad(zt, zp): d = delta_z(zt, zp); return float(1.4826*np.median(np.abs(d-np.median(d))))
def outlier_rate(zt, zp, thr=0.05): return float(np.mean(np.abs(delta_z(zt, zp)) > thr))

In [ ]:
# --- build train/val matrices ---
base_c = ["base_RF", "base_HGB", "base_MLP"]
mask_c = [c for c in df.columns if c.startswith("mask_")]
emb_c  = sorted([c for c in df.columns if c.startswith("emb_")], key=lambda c: int(c[4:]))
feat = base_c + mask_c + emb_c
def xy(name):
    d = df[df.split == name]
    return (d[feat].to_numpy("float32"), np.log1p(d["redshift"].to_numpy("float32")),
            d["redshift"].to_numpy("float64"), d["n_absent"].to_numpy("int16"))
Xtr, ytr, _, _   = xy("train")
Xva, yva, zva, nva = xy("val")
print("in-dim", len(feat), "| train", Xtr.shape, "| val", Xva.shape)

In [ ]:
# --- train (MDN NLL; early-stop on FULL-feature val sigma_MAD) ---
class SigmaMadCB(tf.keras.callbacks.Callback):
    def __init__(self, X, z): self.X, self.z = X, z
    def on_epoch_end(self, e, logs=None):
        zp = np.expm1(mdn_point(self.model.predict(self.X, verbose=0)))
        sm = sigma_mad(self.z, zp); logs = logs or {}; logs["val50k_sigma_MAD"] = sm
        try:
            if mlflow.active_run(): mlflow.log_metric("val50k_sigma_MAD", float(sm), step=e)
        except Exception: pass
        print(f"  epoch {e}: val50k sigma_MAD={sm:.4f}")

K = 5
es = (nva == 0)                              # full-feature val subset, comparable to the baselines
model = build_fusion(len(feat), hidden=(128,128,64), mdn=K, drop=0.3)
model.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss=mdn_nll(K))
cbs = [SigmaMadCB(Xva[es], zva[es]),
       tf.keras.callbacks.EarlyStopping("val50k_sigma_MAD", mode="min", patience=10, restore_best_weights=True),
       tf.keras.callbacks.ReduceLROnPlateau("val50k_sigma_MAD", mode="min", factor=0.5, patience=3, min_lr=1e-5)]
mlflow.start_run(run_name="fusion-v4")
mlflow.log_params(dict(mdn=K, hidden="128-128-64", lr=3e-4, batch=1024, drop=0.3,
                       inputs="3base+16mask+64emb", n_train=len(Xtr), n_val=len(Xva)))
model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=80, batch_size=1024, callbacks=cbs)

In [ ]:
# --- val sigma_MAD by missing bucket + log + save (best weights already restored in-memory) ---
zp = np.expm1(mdn_point(model.predict(Xva, verbose=0)))
def bucket(lab, mk):
    if mk.sum():
        sm, ou = sigma_mad(zva[mk], zp[mk]), outlier_rate(zva[mk], zp[mk])*100
        print(f"  {lab:12s} n={int(mk.sum()):6d}  sigma_MAD={sm:.5f}  outlier={ou:.2f}%")
        mlflow.log_metric("val_%s_sMAD" % lab.replace(" ","_").replace("=",""), float(sm))
print("=== val sigma_MAD by missing bucket ===")
bucket("all",        np.ones(len(nva), bool))
bucket("absent=0",   nva == 0)
bucket("absent 1-2", (nva >= 1) & (nva <= 2))
bucket("absent 3-5", (nva >= 3) & (nva <= 5))
bucket("absent 6+",  nva >= 6)
# the full-feature val subset (n_absent==0) == the 45k val_objids -> log as val50k_sigma_MAD (CNN-comparable)
mlflow.log_metric("val50k_sigma_MAD", sigma_mad(zva[nva == 0], zp[nva == 0]))
mlflow.log_metric("val50k_outlier", outlier_rate(zva[nva == 0], zp[nva == 0]))
model.save("/content/fusion_v4.keras")                   # best weights (restore_best_weights=True)
mlflow.log_artifact("/content/fusion_v4.keras")
mlflow.end_run()
print("done — best-val50k model logged to MLflow (experiment 'fusion')")